# 21 · RAG Evaluation

**Idea:** you cannot improve what you do not measure. These are LLM-as-judge versions of the metrics popularized by RAGAS, computed by hand so the mechanics are visible:

- **context_precision** — fraction of *retrieved* chunks that are actually relevant.
- **context_recall** — fraction of the *ground-truth* answer supported by the context.
- **faithfulness** — is the generated answer entailed by the context (not invented)?
- **answer_relevance** — does the answer actually address the question?

Each returns a 0–1 score from a single judge call, so any provider works and token budgets stay tiny.

> Judge runs on Ollama by default. One cell optionally judges with OpenAI — skips if no key.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

Produce something to grade: retrieve context and generate an answer with the standard `rag_answer` pipeline.

In [ ]:
from rag_lab import load_markdown, NumpyStore, rag_answer
from study_buddy import CostTracker

store = NumpyStore(provider='ollama'); store.add(load_markdown())
question = 'What is hybrid search and why combine BM25 with vector search?'
ground_truth = 'Hybrid search combines keyword (BM25) and semantic vector search, fusing their rankings so it catches both exact term matches and meaning-based matches.'

tracker = CostTracker()
res = rag_answer(question, store, provider='ollama', k=4, tracker=tracker)
print(res.answer)

Run the full metric suite on this `(question, answer, context)` example. `evaluate` scores all four metrics with one judge.

In [ ]:
from rag_lab import evaluate

result = evaluate(
    question, res.answer, res.hits,
    ground_truth=ground_truth,
    provider='ollama', tracker=tracker,
)
print(result.summary())
print('\njudge cost:', tracker.summary())

Sanity check the metrics: an **off-topic** retrieval should score low `context_precision`. If a bad pipeline still scored high, the metric would be useless.

In [ ]:
from rag_lab import Judge, context_precision

bad_hits = store.search('what is the capital of France?', k=4)
judge = Judge(provider='ollama')
print('precision on off-topic retrieval:', round(context_precision(judge, question, bad_hits), 2))

Optionally use OpenAI as the judge — a stronger judge gives more reliable scores. Skips if no key.

In [ ]:
try:
    cloud = CostTracker()
    r = evaluate(question, res.answer, res.hits, ground_truth=ground_truth,
                 provider='openai', model='gpt-4o-mini', tracker=cloud)
    print(r.summary())
    print('cost:', cloud.summary())
except Exception as e:
    print('openai judge skipped:', type(e).__name__, e)

**Takeaway:** these four scores separate *retrieval* failures (low precision/recall) from *generation* failures (low faithfulness/relevance), so you know which half of the pipeline to fix. This harness is the backbone of the capstone's evaluation.

Next up → `22_contextual_retrieval.ipynb`, one technique that directly moves `context_precision`.